# Optional agent alignment and prompt optimization lab

This connected, disabled-by-default lab adapts MLflow's [agent alignment and optimization cookbook](https://mlflow.org/cookbook/agent-alignment-optimization/) to the repository's stricter release lifecycle.

The safe sequence is:

1. calibrate a domain judge against human feedback;
2. validate the frozen judge on held-out labels;
3. optimize an exact seed prompt against a separate training split and bounded request budget;
4. register the optimized text as a new immutable prompt version;
5. evaluate that exact version on a final held-out release split;
6. let the ordinary release gate decide `adopt`, `reject`, or `inconclusive`.

This notebook never moves `production`. Optimization proposes a change; it does not authorize deployment.

## 1. Lock three disjoint evidence splits

Judge calibration, optimizer training, and final release testing must not reuse cases. Otherwise the aligned judge and optimized prompt can certify the same examples they learned from.

In [ ]:
import hashlib
import json

SPLIT_MANIFEST = {
    "judge_calibration": [
        "cal-revenue-01",
        "cal-margin-01",
        "cal-guidance-01",
        "cal-cash-01",
        "cal-risk-01",
        "cal-policy-01",
    ],
    "optimizer_training": [
        "train-revenue-01",
        "train-margin-01",
        "train-guidance-01",
        "train-cash-01",
        "train-risk-01",
        "train-policy-01",
    ],
    "held_out_release": [
        "holdout-revenue-01",
        "holdout-margin-01",
        "holdout-guidance-01",
        "holdout-cash-01",
        "holdout-risk-01",
        "holdout-policy-01",
    ],
}

TOPIC_FIXTURES = {
    "revenue": {
        "question": "What was fictional quarterly revenue?",
        "earnings_excerpt": "Revenue was $128.4 million, up 12%.",
        "source_id": "ARS-FY25-Q2-RESULTS",
        "required_fact": "$128.4 million",
    },
    "margin": {
        "question": "What was fictional operating margin?",
        "earnings_excerpt": "Operating margin was 18.6% versus 16.9%.",
        "source_id": "ARS-FY25-Q2-RESULTS",
        "required_fact": "18.6%",
    },
    "guidance": {
        "question": "What fictional revenue guidance was supplied?",
        "earnings_excerpt": "Revenue guidance was $132 million to $136 million.",
        "source_id": "ARS-FY25-Q2-GUIDANCE",
        "required_fact": "$132 million to $136 million",
    },
    "cash": {
        "question": "What was fictional free cash flow?",
        "earnings_excerpt": "Free cash flow was $21.7 million.",
        "source_id": "ARS-FY25-Q2-CASH-RISK",
        "required_fact": "$21.7 million",
    },
    "risk": {
        "question": "What fictional supplier risk was disclosed?",
        "earnings_excerpt": "Single-source supplier concentration is a risk.",
        "source_id": "ARS-FY25-Q2-CASH-RISK",
        "required_fact": "single-source supplier concentration",
    },
    "policy": {
        "question": "Should I buy shares based on this fictional excerpt?",
        "earnings_excerpt": "The excerpt contains historical fictional results only.",
        "source_id": "ARS-FY25-Q2-RESULTS",
        "required_fact": "cannot provide investment advice",
    },
}


def split_record(case_id):
    topic = case_id.split("-")[1]
    fixture = TOPIC_FIXTURES[topic]
    expected = (
        f"{fixture['required_fact']} [source: {fixture['source_id']}]"
    )
    return {
        "inputs": {
            "question": fixture["question"],
            "earnings_excerpt": fixture["earnings_excerpt"],
            "source_id": fixture["source_id"],
        },
        "outputs": expected,
        "expectations": {
            "required_facts": [fixture["required_fact"]],
            "source_id": fixture["source_id"],
            "no_investment_recommendation": True,
        },
    }


SPLIT_RECORDS = {
    split: [split_record(case_id) for case_id in case_ids]
    for split, case_ids in SPLIT_MANIFEST.items()
}

split_sets = {name: set(case_ids) for name, case_ids in SPLIT_MANIFEST.items()}
assert split_sets["judge_calibration"].isdisjoint(split_sets["optimizer_training"])
assert split_sets["judge_calibration"].isdisjoint(split_sets["held_out_release"])
assert split_sets["optimizer_training"].isdisjoint(split_sets["held_out_release"])
assert sum(len(case_ids) for case_ids in split_sets.values()) == len(
    set().union(*split_sets.values())
)

SPLIT_MANIFEST_DIGEST = hashlib.sha256(
    json.dumps(
        SPLIT_MANIFEST,
        ensure_ascii=True,
        separators=(",", ":"),
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()
{
    "split_manifest_digest": SPLIT_MANIFEST_DIGEST,
    "case_counts": {name: len(case_ids) for name, case_ids in split_sets.items()},
}

## 2. Make experimental dependencies and spend bounds explicit

`MemAlignOptimizer` is experimental and requires DSPy. `GepaPromptOptimizer` requires GEPA. Neither dependency is present in this repository's certified locks, so the connected lab must remain off unless dependency policy, exact locks, template locks, and compatibility evidence are updated together.

In [ ]:
import importlib.util

OPTIMIZATION_BUDGET = {
    "max_metric_calls": 30,
    "max_training_cases": len(SPLIT_MANIFEST["optimizer_training"]),
    "evaluation_concurrency": 1,
    "request_timeout_seconds": 60,
}
EXPERIMENTAL_DEPENDENCIES = {
    "dspy": importlib.util.find_spec("dspy") is not None,
    "gepa": importlib.util.find_spec("gepa") is not None,
}
{
    "optimization_budget": OPTIMIZATION_BUDGET,
    "experimental_dependencies": EXPERIMENTAL_DEPENDENCIES,
    "ready": all(EXPERIMENTAL_DEPENDENCIES.values()),
}

## 3. Connected alignment and optimization skeleton

Before enabling the next cell:

- collect at least the approved number of balanced human labels with rationales and source `group:domain-reviewers`;
- use the same assessment name for human feedback and the judge;
- freeze and validate the aligned judge before optimization;
- set the exact seed prompt, optimizer data, governed reflection-model URI, and validated judge version/run evidence;
- make `predict_fn` load and format the registered seed prompt **inside every call**. Building an agent once around the baseline text would make optimizer candidates inert.

In [ ]:
PERSIST_EVIDENCE_TO_DATABRICKS = False
RUN_EXPERIMENTAL_OPTIMIZATION = False
JUDGE_NAME = "uncertainty_explanation"
JUDGE_EXPERIMENT_ID = None
SEED_PROMPT_URI = None  # Exact URI: prompts:/<qualified-name>/<version>
REFLECTION_MODEL_URI = None  # Resolve a governed logical model keylessly.
ALIGNED_JUDGE_VERSION = None
JUDGE_VALIDATION_RUN_ID = None
JUDGE_VALIDATION_AGREEMENT = None
JUDGE_VALIDATION_LABEL_COUNT = None

if PERSIST_EVIDENCE_TO_DATABRICKS or RUN_EXPERIMENTAL_OPTIMIZATION:
    from functools import partial

    import mlflow
    from mlflow import MlflowClient

    from aai_core.experiments import (
        ExperimentManager,
        ExperimentRunMetadata,
        RunPurpose,
    )
    from examples.notebook_setup import (
        get_or_create_uc_evaluation_dataset,
        preflight_databricks,
        preflight_databricks_evidence,
        prepare_notebook_environment,
    )

    environment = prepare_notebook_environment(
        evidence_destination="databricks"
    )
    evidence = preflight_databricks_evidence(environment)
    connected = (
        preflight_databricks(environment)
        if RUN_EXPERIMENTAL_OPTIMIZATION
        else None
    )
    datasets = {
        split: get_or_create_uc_evaluation_dataset(
            evidence=evidence,
            dataset_name=f"fictional_{split}_v1",
            records=SPLIT_RECORDS[split],
            mlflow_module=mlflow,
        )
        for split in SPLIT_MANIFEST
    }
    experiments = ExperimentManager(
        experiment_name=evidence.experiment_name,
        context=evidence.context.tags,
    )
    client = MlflowClient()

    with experiments.run(
        run_name="agent-alignment-governed-evidence",
        description=(
            "Governed registration of disjoint judge-calibration, optimizer-"
            "training, and held-out release datasets. Optimization remains "
            "experimental and cannot move a production alias."
        ),
        parameters={
            "split_manifest_digest_sha256": SPLIT_MANIFEST_DIGEST,
            "optimization_enabled": RUN_EXPERIMENTAL_OPTIMIZATION,
        },
        metadata=ExperimentRunMetadata(
            purpose=RunPurpose.RESULT,
            change_id="agent-alignment-optimization-v1",
            change_summary="Optimize one prompt with disjoint governed evidence.",
        ),
    ) as optimization_run:
        for split, dataset in datasets.items():
            mlflow.log_input(dataset, context=split)

        if RUN_EXPERIMENTAL_OPTIMIZATION:
            required_values = {
                "JUDGE_EXPERIMENT_ID": JUDGE_EXPERIMENT_ID,
                "SEED_PROMPT_URI": SEED_PROMPT_URI,
                "REFLECTION_MODEL_URI": REFLECTION_MODEL_URI,
                "ALIGNED_JUDGE_VERSION": ALIGNED_JUDGE_VERSION,
                "JUDGE_VALIDATION_RUN_ID": JUDGE_VALIDATION_RUN_ID,
                "JUDGE_VALIDATION_AGREEMENT": JUDGE_VALIDATION_AGREEMENT,
                "JUDGE_VALIDATION_LABEL_COUNT": JUDGE_VALIDATION_LABEL_COUNT,
            }
            missing_values = [
                name
                for name, value in required_values.items()
                if value is None
                or (isinstance(value, str) and not value.strip())
            ]
            if missing_values:
                raise ValueError(
                    f"Configure the connected lab first: {missing_values}"
                )
            if not all(EXPERIMENTAL_DEPENDENCIES.values()):
                raise RuntimeError(
                    "DSPy and GEPA are not in the certified locks; complete the "
                    "dependency policy and compatibility workflow first"
                )
            if (
                JUDGE_VALIDATION_AGREEMENT < 0.75
                or JUDGE_VALIDATION_LABEL_COUNT < 50
            ):
                raise RuntimeError(
                    "The aligned judge has not passed held-out agreement and "
                    "sample-size requirements"
                )

            from mlflow.genai.optimize import GepaPromptOptimizer
            from mlflow.genai.scorers import get_scorer

            from aai_core.tracing import (
                TraceCaptureMode,
                TraceIntegration,
                TracePolicy,
                configure_tracing,
                set_trace_resource_context,
            )

            configure_tracing(
                connected.context.tags,
                experiment_name=connected.experiment_name,
                integration=TraceIntegration.SDK,
                policy=TracePolicy(capture_mode=TraceCaptureMode.FULL),
            )
            model = connected.model
            aligned_judge = get_scorer(
                name=JUDGE_NAME,
                experiment_id=JUDGE_EXPERIMENT_ID,
                version=int(ALIGNED_JUDGE_VERSION),
            )
            seed_prompt = mlflow.genai.load_prompt(SEED_PROMPT_URI)
            client.link_prompt_version_to_run(
                optimization_run.info.run_id,
                seed_prompt,
            )
            trace_ids_by_prompt = {}

            def predict_with_prompt_uri(
                prompt_uri,
                question,
                earnings_excerpt,
                source_id,
            ):
                active_prompt = mlflow.genai.load_prompt(prompt_uri)
                rendered = active_prompt.format(
                    question=question,
                    earnings_excerpt=earnings_excerpt,
                    source_id=source_id,
                )
                with mlflow.start_span(
                    name="earnings_summary.optimization_prediction",
                    span_type="CHAIN",
                ) as application_span:
                    set_trace_resource_context(connected.context.tags)
                    application_span.set_attribute(
                        "mlflow.message.format",
                        "openai",
                    )
                    application_span.set_inputs(
                        {
                            "messages": [
                                {"role": "user", "content": rendered}
                            ]
                        }
                    )
                    mlflow.update_current_trace(request_preview=question)
                    response = model.generate(
                        [{"role": "user", "content": rendered}],
                        temperature=0.0,
                        max_tokens=400,
                    )
                    application_span.set_outputs(
                        {"content": response.content}
                    )
                    mlflow.update_current_trace(
                        response_preview=response.content
                    )
                mlflow.flush_trace_async_logging()
                trace_id = mlflow.get_last_active_trace_id()
                if trace_id is None:
                    raise RuntimeError("Prediction did not produce a trace")
                client.link_prompt_versions_to_trace(
                    prompt_versions=[active_prompt],
                    trace_id=trace_id,
                )
                trace_ids_by_prompt.setdefault(prompt_uri, []).append(trace_id)
                return response.content

            def predict_with_registered_prompt(
                question,
                earnings_excerpt,
                source_id,
            ):
                return predict_with_prompt_uri(
                    SEED_PROMPT_URI,
                    question,
                    earnings_excerpt,
                    source_id,
                )

            optimization_result = mlflow.genai.optimize_prompts(
                predict_fn=predict_with_registered_prompt,
                train_data=datasets["optimizer_training"],
                prompt_uris=[SEED_PROMPT_URI],
                optimizer=GepaPromptOptimizer(
                    reflection_model=REFLECTION_MODEL_URI,
                    max_metric_calls=OPTIMIZATION_BUDGET["max_metric_calls"],
                    display_progress_bar=True,
                ),
                scorers=[aligned_judge],
            )
            optimized_prompt = optimization_result.optimized_prompts[0]
            client.link_prompt_version_to_run(
                optimization_run.info.run_id,
                optimized_prompt,
            )

            heldout_results = {}
            for role, prompt_version in (
                ("baseline", seed_prompt),
                ("optimized", optimized_prompt),
            ):
                with experiments.run(
                    run_name=f"agent-alignment-heldout-{role}",
                    description=(
                        f"Observed held-out release evidence for the exact {role} "
                        "prompt version; no alias is moved by this run."
                    ),
                    nested=True,
                    metadata=ExperimentRunMetadata(
                        purpose=(
                            RunPurpose.BASELINE
                            if role == "baseline"
                            else RunPurpose.CHANGE
                        ),
                        change_id="agent-alignment-optimization-v1",
                        change_summary="Evaluate optimized prompt on held-out data.",
                    ),
                ) as heldout_run:
                    mlflow.log_input(
                        datasets["held_out_release"],
                        context="held_out_release",
                    )
                    client.link_prompt_version_to_run(
                        heldout_run.info.run_id,
                        prompt_version,
                    )
                    heldout_results[role] = mlflow.genai.evaluate(
                        data=datasets["held_out_release"],
                        predict_fn=partial(
                            predict_with_prompt_uri,
                            prompt_version.uri,
                        ),
                        scorers=[aligned_judge],
                    )

            metric_name = f"{aligned_judge.name}/mean"
            heldout_scores = {
                role: result.metrics.get(metric_name)
                for role, result in heldout_results.items()
            }
            if any(score is None for score in heldout_scores.values()):
                raise RuntimeError(
                    f"Held-out results lack required metric {metric_name!r}"
                )
            decision = (
                "adopt"
                if heldout_scores["optimized"] >= heldout_scores["baseline"]
                else "reject"
            )
            client.set_tag(
                optimization_run.info.run_id,
                "aai.decision",
                decision,
            )
            print(
                {
                    "initial_score": optimization_result.initial_eval_score,
                    "final_score": optimization_result.final_eval_score,
                    "optimized_prompt_uri": optimized_prompt.uri,
                    "heldout_scores": heldout_scores,
                    "linked_trace_ids": trace_ids_by_prompt,
                    "decision": decision,
                    "alias_moved": False,
                }
            )
        else:
            print(
                {
                    "run_id": optimization_run.info.run_id,
                    "datasets": {
                        split: dataset.name
                        for split, dataset in datasets.items()
                    },
                    "optimization": "skipped",
                }
            )
else:
    print("DATABRICKS EVIDENCE AND EXPERIMENTAL OPTIMIZATION SKIPPED")

## 4. Register, test held-out cases, then use the normal gate

If optimization produces a useful template:

1. register it as a new immutable version of the same qualified prompt;
2. record its content digest, exact URI, seed URI, split-manifest digest, optimizer/reflection model, dependency digest, request budget, and source commit;
3. load that exact new version inside every prediction on `held_out_release`;
4. run deterministic fact, citation, policy, critical-row, latency, token, cost, and cost-coverage gates, with the validated judge as additional evidence;
5. compare against the untouched baseline and choose `adopt`, `reject`, or `inconclusive`;
6. move a controlled alias only in the ordinary release workflow after all checks pass.

Do not inspect private judge fields such as `_semantic_memory`; record public versioned instructions and alignment evidence.

In [ ]:
{
    "stage": "optimization_plan",
    "split_manifest_digest": SPLIT_MANIFEST_DIGEST,
    "experimental_dependencies_ready": all(EXPERIMENTAL_DEPENDENCIES.values()),
    "decision": "inconclusive",
    "release": "blocked",
    "reason": (
        "optimization is disabled and cannot authorize release; register any "
        "proposed prompt and run the final held-out gate"
    ),
}